### TEST 02D · PVAR con presión + gradiente REAL en el estado (dominio pequeño)
Estado del PVAR: 20 PCs de presión + 10 PCs del gradiente real (del 04D). Así la relación presión–gradiente es una correlación lineal dentro del estado, que el VAR reproduce por construcción.

- Las PCs del gradiente son muy asimétricas → PIT empírico a normal antes del HarmonicStandardizer, y al revés después. Las de presión entran como en el 02A.
- p=10, n_harm=1, choques bootstrap (como h1b).
- Guarda en `outputs/var_6h_wmed_g/` (nombre de variante `g20b`). Las simulaciones llevan ya presión y gradiente: no pasan por el 03.

**Limitación del test:** la PIT empírica no extrapola, las PCs del gradiente simuladas no pasan del máximo histórico.

In [1]:
import os, glob, json, pickle, time

import pandas as pd
import numpy as np
from scipy.signal import find_peaks
from xgboost import XGBRegressor

from utils.tv_var.harmonic_standardizer import HarmonicStandardizer
from utils.tv_var.periodic_var import PeriodicVAR
from utils.pit.piece_wise_marginal import PiecewiseMarginal, to_normal, from_normal
from utils.pit.empirical_marginal import EmpiricalPiece

#### Load Data

In [2]:
N_MSL = 20
pcs_msl = pd.read_csv('outputs/pca/pcs_msl_30_6h_wmed.csv', index_col=0, parse_dates=True).iloc[:, :N_MSL]
pcs_greal = pd.read_csv('outputs/grad_rec/pcs_grad_real_6h_wmed.csv', index_col=0, parse_dates=True)

estado = pcs_msl.join(pcs_greal, how='inner')
cols_msl, cols_gr = list(pcs_msl.columns), list(pcs_greal.columns)
print(estado.shape, estado.index[0], estado.index[-1])

(126864, 30) 1940-01-01 00:00:00 2026-10-31 18:00:00


#### Gradiente: PIT empírica → normal

In [3]:
marg_gr = {c: PiecewiseMarginal([EmpiricalPiece()], []).fit(estado[c]) for c in cols_gr}

estado_n = estado.copy()
for c in cols_gr:
    estado_n[c] = to_normal(marg_gr[c].pit())

def gr_a_fisico(df):
    df = df.copy()
    for c in cols_gr:
        df[c] = marg_gr[c].ppf(from_normal(df[c].values))
    return df

pd.DataFrame({'asimetría antes': estado[cols_gr].skew(), 'asimetría después': estado_n[cols_gr].skew()}).round(2)

,asimetría antes,asimetría después
grad_PC1,2.13,0.0
grad_PC2,0.32,-0.0
grad_PC3,0.50,-0.0
grad_PC4,-0.61,0.0
grad_PC5,0.23,-0.0
grad_PC6,0.30,-0.0
grad_PC7,-0.19,0.0
grad_PC8,0.80,0.0
grad_PC9,0.53,-0.0
grad_PC10,0.45,0.0


#### Estandarización armónica + PVAR

In [4]:
harm_std = HarmonicStandardizer(n_harmonics=3)
harm_std.fit(estado_n)
estado_z = harm_std.transform(estado_n)

var = PeriodicVAR(p=10, n_harm=1, freq="6h", noise_draw="bootstrap").fit(estado_z)
print(var.stability())
print(var.n_params(), var.bic())

{'rho': 0.016196197952416575, 'rho_daily': 0.9887677261574126, 'rho_step': 0.9971800255649804, 'stable': True}
32670 2872129.4891415425


#### Simulaciones

In [5]:
nombre = 'g20b'
out = 'outputs/var_6h_wmed_g'
sim_dir = f'{out}/simulaciones_{nombre}'
os.makedirs(sim_dir, exist_ok=True)
pickle.dump({'var': var, 'harm_std': harm_std, 'marg_gr': marg_gr, 'cols_msl': cols_msl, 'cols_gr': cols_gr},
            open(f'{out}/var_{nombre}.pkl', 'wb'))

burn = int(pd.Timedelta(days=3*365) / pd.Timedelta("6h"))
fechas = pd.date_range("1940-01-01", "2026-12-31 18:00", freq="6h")
N_SIMS = 20

def ruido_bootstrap(var, fechas, seed):
    rng = np.random.default_rng(seed)
    eps = np.empty((len(fechas), var.k_))
    for m, pool in var.resid_month_.items():
        sel = fechas.month == m
        eps[sel] = pool[rng.integers(len(pool), size=sel.sum())]
    return pd.DataFrame(eps, index=fechas, columns=var.columns_)

for i in range(N_SIMS):
    f_sim = f'{sim_dir}/sim_{i:03d}.parquet'
    if os.path.exists(f_sim):
        continue
    t = time.time()
    sim_z = var.simulate(noise=ruido_bootstrap(var, fechas, seed=i), burn=burn)
    gr_a_fisico(harm_std.inverse_transform(sim_z)).astype('float32').to_parquet(f_sim)
    print(f'{nombre} sim {i:03d}  ({time.time() - t:.0f} s)')

g20b sim 000  (4 s)
g20b sim 001  (2 s)
g20b sim 002  (2 s)
g20b sim 003  (2 s)
g20b sim 004  (2 s)
g20b sim 005  (2 s)
g20b sim 006  (2 s)
g20b sim 007  (2 s)
g20b sim 008  (2 s)
g20b sim 009  (2 s)
g20b sim 010  (2 s)
g20b sim 011  (2 s)
g20b sim 012  (2 s)
g20b sim 013  (2 s)
g20b sim 014  (2 s)
g20b sim 015  (2 s)
g20b sim 016  (2 s)
g20b sim 017  (2 s)
g20b sim 018  (2 s)
g20b sim 019  (2 s)


#### Validación rápida del estado: σ, asimetría y corr presión–gradiente

In [6]:
sims = [pd.read_parquet(f).astype('float64') for f in sorted(glob.glob(f'{sim_dir}/sim_*.parquet'))]
sims = [s.loc['1941':'2025'] for s in sims]
ref = estado.loc['1941':'2025']

def corr_pg(d):                                           # corr máxima en valor absoluto de cada PC de gradiente con las de presión
    c = d.corr().loc[cols_gr, cols_msl]
    return c.abs().max(axis=1)

pd.DataFrame({'σ sim/hist': pd.concat([s.std() for s in sims], axis=1).median(axis=1) / ref.std(),
              'asim hist': ref.skew(),
              'asim sim': pd.concat([s.skew() for s in sims], axis=1).median(axis=1)}).loc[cols_gr].join(
    pd.DataFrame({'|corr| máx con presión hist': corr_pg(ref),
                  'sim': pd.concat([corr_pg(s) for s in sims], axis=1).median(axis=1)})).round(3)

,σ sim/hist,asim hist,asim sim,|corr| máx con presión hist,sim
grad_PC1,1.160,2.135,2.925,0.409,0.357
grad_PC2,1.047,0.325,-0.534,0.520,0.506
grad_PC3,1.123,0.499,0.300,0.381,0.356
grad_PC4,1.151,-0.633,-2.990,0.342,0.293
grad_PC5,1.103,0.222,1.139,0.345,0.320
grad_PC6,1.117,0.227,1.214,0.217,0.207
grad_PC7,1.128,-0.191,-0.646,0.256,0.242
grad_PC8,1.067,0.801,1.695,0.257,0.245
grad_PC9,1.059,0.531,0.980,0.253,0.252
grad_PC10,1.050,0.451,0.639,0.310,0.280


#### ¿Vuelven los picos de f? XGB `p20_g10` del 04D sobre histórico y simulaciones

In [7]:
xgb_dir = 'outputs/xgb_p102_wmed_p20_g10'
xgb = XGBRegressor()
xgb.load_model(f'{xgb_dir}/xgb_final.json')
meta = json.load(open(f'{xgb_dir}/xgb_features.json'))
lags = meta['lags_h']

def f_desde(d):
    Xs = pd.concat({f'l{l}': d.shift(freq=pd.Timedelta(hours=l)) for l in lags}, axis=1).reindex(d.index)
    Xs.columns = [f'{c}_l{l[1:]}' for l, c in Xs.columns]
    Xs = Xs.dropna()[meta['columns']]
    return pd.Series(xgb.predict(Xs), index=Xs.index)

f_hist = f_desde(estado).loc['1941':'2025']                  # mismo periodo que el ajuste del PVAR
f_sims = [f_desde(s) for s in sims]
umbral = f_hist.quantile(0.99)

def resumen(f):
    pk = find_peaks(f.values, height=umbral, distance=12)[0]
    return {'q90': f.quantile(0.90), 'q99': f.quantile(0.99), 'q99.9': f.quantile(0.999),
            'máx anual medio': f.groupby(f.index.year).max().mean(),
            'picos f > q99 hist / década': len(pk) * 10 / f.index.year.nunique()}

h = pd.Series(resumen(f_hist))
pd.DataFrame({'histórico': h, 'g20b / hist': pd.DataFrame([resumen(f) for f in f_sims]).median() / h}).round(3)

,histórico,g20b / hist
q90,1.047,0.834
q99,1.902,0.802
q99.9,2.448,0.812
máx anual medio,2.456,0.813
picos f > q99 hist / década,48.353,0.208


In [8]:
est = lambda m: np.select([np.isin(m, [12, 1, 2]), np.isin(m, [3, 4, 5]), np.isin(m, [6, 7, 8])], ['invierno', 'primavera', 'verano'], 'otoño')

def picos_est(f):
    pk = f.index[find_peaks(f.values, height=umbral, distance=12)[0]]
    return pd.Series(est(pk.month)).value_counts() * 10 / f.index.year.nunique()

tabla_est = pd.DataFrame({'histórico': picos_est(f_hist),
                          'g20b': pd.concat([picos_est(f) for f in f_sims], axis=1).fillna(0).median(axis=1)})
tabla_est['g20b/hist'] = tabla_est['g20b'] / tabla_est['histórico']
tabla_est.loc[['invierno', 'primavera', 'verano', 'otoño']].round(2)

,histórico,g20b,g20b/hist
invierno,19.06,5.41,0.28
primavera,13.65,2.29,0.17
verano,5.41,1.06,0.20
otoño,10.24,1.35,0.13


In [9]:
from bluemath_tk.core.io import load_model
import xarray as xr

pca_grad = load_model('outputs/grad_rec/pca_grad_wmed.pkl')

def frac_negativa(pcs_gr, n=2000, seed=0):
    p = pcs_gr.sample(n, random_state=seed)
    v = np.zeros((n, 50)); v[:, :len(cols_gr)] = p.values          # mismas 10 PCs en los dos casos
    da = xr.DataArray(v, dims=('time', 'n_component'), coords={'time': p.index, 'n_component': np.arange(50)})
    campo = pca_grad.inverse_transform(da)['grad2']
    return float((campo < 0).mean())

print(f"hist: {frac_negativa(ref[cols_gr]):.3f} | sim: {np.median([frac_negativa(s[cols_gr]) for s in sims[:5]]):.3f}")

hist: 0.035 | sim: 0.097
